# 로컬 OCR — 저장하고 다음 날 이어서 실행

배치로 OCR을 수행하고, 결과 한 장이 체크포인트 한 개입니다. **한 장마다 JSONL에 기록하고 디스크 동기화**합니다.
다음 날 같은 노트북에서 **1 → 2 → 3 → 5 → 6**을 실행하면 저장된 ID를 건너뜁니다.
선택 보정 셀 4는 실행하지 않아도 됩니다. 마지막에 셀 7로 결과를 확인하세요.

- `RUN_HOURS = 2.0`: 오늘은 약 2시간 동안 실행하고 현재 배치 완료 후 자동 중지.
- `RUN_HOURS = None`: 시간 제한 없이 실행. `MAX_IMAGES_PER_RUN`으로 장수 제한도 가능.
- 즉시 멈추려면 Jupyter/VS Code의 **실행 중지(Interrupt)**를 누르고 저장 메시지를 확인한 뒤 종료.
- 강제 종료/전원 차단 시 처리 중인 이미지나 마지막 미완성 기록은 다음 날 다시 처리할 수 있습니다.
- 절전 중에는 OCR이 진행되지 않습니다. 자동 중지는 컴퓨터 종료 기능이 아닙니다.
- 같은 결과 파일에 두 노트북/프로세스를 동시에 실행하지 마세요.

결과: `output/ocr_cache/train_spotting.jsonl`
진행 요약: `output/ocr_cache/train_spotting.progress.json`
완료 여부는 요약이 아니라 JSONL에서 다시 읽습니다. 기존 로컬 결과도 이어서 사용합니다.

파일 교체 오류 복구: `*_spotting_local_resume.jsonl`이 있으면 해당 파일을 우선 사용합니다.
후속 실험에서도 실제 출력 경로를 사용하세요. 기존 파일에는 새 결과가 추가되지 않습니다.


## 1. 라이브러리

In [ ]:
# Resolve shared code and local assets from the repository root.
import os as _project_os
import sys as _project_sys
from pathlib import Path as _ProjectPath
_project_root = next((p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                      if (p / 'src' / 'ocr_engine.py').is_file()), None)
if _project_root is not None:
    _project_os.chdir(_project_root)
    if str(_project_root / 'src') not in _project_sys.path:
        _project_sys.path.insert(0, str(_project_root / 'src'))
# End repository path setup.

import gc, json, os, sys, time
from pathlib import Path

# Find the project even when the notebook is opened from a nested folder.
PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'src' / 'ocr_engine.py').is_file() and (p / 'data').is_dir()), None)
assert PROJECT_DIR is not None, '프로젝트 폴더에서 노트북을 실행하세요.'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))
for key in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE'):
    os.environ[key] = '1'

import pandas as pd
import torch
from tqdm.auto import tqdm
import ocr_engine as oe
from ocr_checkpoint import durable_append, atomic_json, resume_ids, validate_metadata

print('프로젝트:', PROJECT_DIR)
print('torch', torch.__version__, '/ CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. 파라미터

`RUN_HOURS` 또는 `MAX_IMAGES_PER_RUN`으로 오늘 실행할 분량을 정하세요.
둘 다 설정하면 먼저 도달한 조건으로 멈춥니다. 시간은 모델 로딩 이후부터 계산하며,
배치 처리가 끝난 시점에 검사하므로 마지막 배치 처리 시간만큼 초과할 수 있습니다.
기존 결과를 이어 쓸 때는 `SPLIT`, `PROMPT`, `MAX_NEW_TOKENS`, 모델/픽셀 설정을 유지하세요.
`LIMIT`은 데이터 앞 N장만 대상으로 삼는 시험용 옵션이고, 오늘 실행할 장수는 `MAX_IMAGES_PER_RUN`입니다.

In [ ]:
BATCH_SIZE = 4  # 한 번에 4장. 메모리가 부족하면 자동 축소
SPLIT = 'train'             # train / test / dev / val672
MAX_NEW_TOKENS = 1024
PROMPT = 'Spotting:'
LIMIT = None

RUN_HOURS = 2.0             # 오늘 약 2시간 실행. 제한 없으면 None
MAX_IMAGES_PER_RUN = None  # 오늘 100장만 처리하려면 100
PROGRESS_EVERY = 25        # 진행 요약 파일 갱신 주기. OCR 결과는 매 장 저장
assert RUN_HOURS is None or RUN_HOURS > 0
assert MAX_IMAGES_PER_RUN is None or (isinstance(MAX_IMAGES_PER_RUN, int) and MAX_IMAGES_PER_RUN > 0)
assert isinstance(PROGRESS_EVERY, int) and PROGRESS_EVERY > 0

DATA_DIR = Path('data')
CACHE_DIR = Path('output/ocr_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = CACHE_DIR / f'{SPLIT}_spotting.jsonl'
RESUME_PATH = CACHE_DIR / f'{SPLIT}_spotting_local_resume.jsonl'
if RESUME_PATH.is_file():
    OUT_PATH = RESUME_PATH
PROGRESS_PATH = OUT_PATH.with_suffix('.progress.json')
oe.SPOTTING_PROMPT = PROMPT
print(f'{SPLIT} / {PROMPT!r} / max_new_tokens={MAX_NEW_TOKENS}')
print('출력:', OUT_PATH.resolve())
print('실행 시간 제한:', RUN_HOURS, '시간 / 장수 제한:', MAX_IMAGES_PER_RUN)

## 3. 데이터 로드·검증
CSV를 읽고 이미지가 실제로 존재하는지 먼저 확인합니다.
14시간짜리 작업 도중에 파일 누락을 발견하는 상황을 막기 위한 것입니다.

In [ ]:
VAL672_CSV = Path("output/paddleocr_vl16_halftrain/full/train_half_val.csv")
if SPLIT == "val672":
    assert VAL672_CSV.is_file(), f"실험 A의 분할 파일이 필요합니다: {VAL672_CSV}"
    df = pd.read_csv(VAL672_CSV, keep_default_na=False)
else:
    df = pd.read_csv(DATA_DIR / f"{SPLIT}.csv", keep_default_na=False)
assert df.id.is_unique and df.path.is_unique, "id 또는 path 중복"
missing = [p for p in df.path if not (DATA_DIR / p).is_file()]
assert not missing, f"이미지 누락 {len(missing)}개. 예: {missing[:3]}"
if LIMIT:
    df = df.head(LIMIT).copy()
print(f"{len(df)}장 확인 완료")

## 4. 파라미터 보정 측정 (선택 · GPU 사용)
표본 몇 장을 **넉넉한 상한**으로 돌려, 잘리지 않은 진짜 생성 길이를 잽니다.
그 분포를 보고 `MAX_NEW_TOKENS`를 근거 있게 고릅니다. 전체를 돌리기 전 한 번만 하면 됩니다.

In [ ]:
RUN_CALIBRATION = False    # True로 바꾸면 측정 수행
CAL_N, CAL_CAP = 16, 3072
CAL_PROMPTS = ["Spotting:", "OCR:"]

if not RUN_CALIBRATION:
    print("보정 건너뜀 (RUN_CALIBRATION=False)")
else:
    sample = df.sample(n=min(CAL_N, len(df)), random_state=42)
    model, proc = oe.load_ocr()
    records = []
    try:
        for prompt in CAL_PROMPTS:
            oe.SPOTTING_PROMPT = prompt
            for row in tqdm(sample.to_dict("records"), desc=prompt):
                im = oe.spotting_image(DATA_DIR / row["path"])
                torch.cuda.synchronize(); start = time.perf_counter()
                text, capped, n_tok = oe.run_spotting(model, proc, im, CAL_CAP)
                torch.cuda.synchronize()
                records.append(dict(prompt=prompt, id=row["id"], tokens=n_tok,
                                    truncated=capped, seconds=time.perf_counter() - start,
                                    px=im.width * im.height))
    finally:
        del model, proc
        gc.collect(); torch.cuda.empty_cache()
        oe.SPOTTING_PROMPT = PROMPT      # 파라미터 셀의 값으로 복구

    cal = pd.DataFrame(records)
    (CACHE_DIR / "calibration.json").write_text(
        cal.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")
    for prompt, g in cal.groupby("prompt"):
        q = g.tokens.quantile([0.5, 0.75, 0.9, 1.0]).astype(int)
        print(f"\n[{prompt}] 중앙값 {q[0.5]} / p75 {q[0.75]} / p90 {q[0.9]} / 최대 {q[1.0]}"
              f" / 상한도달 {g.truncated.sum()}/{len(g)}")
        for cap in (512, 1024, 1536, 2048, 3072):
            est = (g.tokens.clip(upper=cap) / g.tokens * g.seconds).mean()
            print(f"   {cap:5d}: {(g.tokens <= cap).mean():5.1%} 커버 "
                  f"/ {len(df)}장 약 {est * len(df) / 3600:5.1f}시간")
    sp = cal[cal.prompt == "Spotting:"]
    print(f"\n이미지 픽셀수 vs 생성 토큰 상관 r = {sp.px.corr(sp.tokens):.3f}"
          " (0에 가까우면 해상도와 무관하다는 뜻)")

## 5. 이어하기 준비

JSONL에 저장된 이미지 ID를 읽습니다. 프롬프트·토큰 한도·메타데이터가 다르면 중단합니다.
강제 종료로 마지막 줄이 미완성이면 원본을 `.bak`으로 보관하고 그 줄만 복구합니다.
같은 OCR 결과의 중복 ID는 백업 후 하나만 남깁니다. 처리 시간 차이는 무시합니다.
파일 중간 손상이나 OCR 결과가 서로 다른 중복 ID는 오류를 표시합니다.

In [ ]:
import importlib
import ocr_checkpoint
importlib.reload(ocr_checkpoint)
from ocr_checkpoint import durable_append, atomic_json, resume_ids, validate_metadata

def prepare_resume():
    settings = {
        'split': SPLIT, 'rows': len(df), 'max_new_tokens': MAX_NEW_TOKENS,
        'prompt': PROMPT, 'model': str(oe.OCR_MODEL_DIR),
        'max_pixels': oe.OCR_MAX_PIXELS,
        'upscale_threshold': oe.SPOTTING_UPSCALE_THRESHOLD,
    }
    validate_metadata(OUT_PATH.with_suffix('.meta.json'), settings)
    done = resume_ids(OUT_PATH, MAX_NEW_TOKENS, PROMPT)
    atomic_json(OUT_PATH.with_suffix('.meta.json'), settings)
    return done, df[~df.id.isin(done)].copy()

done, todo = prepare_resume()
print(f'대상 {len(df)}장 / 완료 {len(df) - len(todo)}장 / 남음 {len(todo)}장')

## 6. 로컬 배치 OCR — 저장 및 이어하기

`BATCH_SIZE = 4`로 여러 장을 함께 처리합니다. 메모리 부족 시 배치를 절반으로 줄여 재시도합니다.
배치 완료 후 결과는 이미지별로 디스크에 저장됩니다. 배치 처리 중 중단하면 저장되지 않은 배치를 다시 처리합니다.
시간 제한은 배치 사이에 검사하므로 마지막 배치 처리 시간만큼 초과할 수 있습니다.
`seconds`는 배치 시간 / 이미지 수이며, 배치 연산에 따른 수치 차이로 단일 처리와 결과가 달라질 수 있습니다.
중지 버튼을 누른 후 `저장 완료`를 확인하고 종료하세요.


In [ ]:
import importlib
import ocr_local_runner
importlib.reload(oe)
importlib.reload(ocr_local_runner)

# Always re-read saved IDs before running.
done, todo = prepare_resume()
ocr_local_runner.run_local(
    df, DATA_DIR, OUT_PATH, MAX_NEW_TOKENS, PROMPT,
    batch_size=BATCH_SIZE, run_hours=RUN_HOURS,
    max_images=MAX_IMAGES_PER_RUN, progress_every=PROGRESS_EVERY,
)


## 7. 캐시 점검 (GPU 불필요)
캐시가 split 전체를 덮는지, 잘림과 빈 결과가 얼마나 되는지 확인합니다.
1·2·3번 셀만 실행한 뒤 이 셀로 바로 와도 됩니다.

In [ ]:
cache = pd.read_json(OUT_PATH, lines=True)
print(f"캐시 {len(cache)}행 / 대상 {len(df)}행")
assert cache.id.is_unique, "캐시에 id 중복"
missing_ids = set(df.id) - set(cache.id)
print("누락:", len(missing_ids), "" if not missing_ids else list(missing_ids)[:3])

# 컬럼명 empty는 DataFrame.empty 속성과 충돌하므로 is_empty를 씁니다.
cache["is_empty"] = cache.ocr_raw.astype(str).str.strip() == ""
print(f"\n빈 결과       {cache['is_empty'].sum():5d}장 ({cache['is_empty'].mean():.1%})")
print(f"길이 제한 도달 {cache['ocr_truncated'].sum():5d}장 ({cache['ocr_truncated'].mean():.1%})")
print(f"\n생성 토큰 분포:\n{cache.ocr_tokens.describe().round(1).to_string()}")
print(f"\n총 소요 {cache.seconds.sum() / 3600:.2f}시간 / 장당 평균 {cache.seconds.mean():.2f}초")
if missing_ids:
    print(f"남은 {len(missing_ids)}장 예상: 약 {cache.seconds.mean() * len(missing_ids) / 3600:.1f}시간")
print("\n샘플 2건:")
for _, r in cache.head(2).iterrows():
    print(f"  {r.id} ({r.ocr_tokens}토큰): {str(r.ocr_raw)[:200]}")

## 이후 실험에서 읽는 법

```python
import pandas as pd
cache = pd.read_json("output/ocr_cache/val672_spotting.jsonl", lines=True)
evidence = dict(zip(cache.id, cache.ocr_raw))     # id -> OCR 원문

val = pd.read_csv("output/paddleocr_vl16_halftrain/full/train_half_val.csv",
                  keep_default_na=False)
val["ocr"] = val.id.map(evidence)
```

**저장 형식(JSONL)** — 한 줄 = JSON 하나 = 이미지 한 장:

| 필드 | 내용 |
|---|---|
| `id`, `path` | 이미지 식별자 |
| `ocr_raw` | OCR 원문. Spotting이면 `글자<\|LOC_n\|>×8`이 줄마다 붙습니다 |
| `ocr_tokens` | 생성 토큰 수 |
| `ocr_truncated` | 길이 제한 도달 여부 |
| `max_new_tokens`, `prompt` | 이 줄을 만든 설정 |
| `seconds` | 소요 시간 |
| `image_width/height` | 전처리 후 크기 |

`ocr_raw`에 원문을 그대로 두므로 좌표 제거·재포맷 같은 후처리는 나중에 얼마든지 가능합니다.

기존 `output/ocr_spotting_val672.jsonl`은 실험 A 산출물로 **512토큰 기준**입니다.
이 캐시(1024)와 설정이 다르니 섞어 쓰지 마세요.